# Quantum Circuit Sensitivity Analyzer

A Qiskit-based simulation study of how gate perturbations and depolarizing noise affect quantum measurement distributions.

**Goal:** establish a baseline circuit, introduce controlled gate changes, simulate the circuits, and quantify how much the output distribution changes.

## 1. Setup

The reusable implementation lives in `src/sensitivity_analyzer.py`. The notebook is the experiment and visualization layer.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram

sys.path.append(str(Path.cwd() / 'src'))
from sensitivity_analyzer import (
    build_experiments, analyze_experiments, create_noise_levels,
    DEFAULT_SHOTS,
)

SHOTS = DEFAULT_SHOTS
ANGLE = np.pi / 4
SEED = 42


## 2. Build the experiment set

In [ ]:
experiments = build_experiments(angle=ANGLE)
for name, circuit in experiments:
    print(f'\n{name}')
    display(circuit.draw('mpl'))


## 3. Ideal simulation

Each circuit is simulated with a fixed seed and the same number of shots so that the comparison is reproducible.

In [ ]:
ideal_rows = analyze_experiments(experiments, shots=SHOTS, seed=SEED)
ideal_df = pd.DataFrame([
    {k: v for k, v in row.items() if k not in ['counts', 'circuit_object']}
    for row in ideal_rows
])
ideal_df


## 4. Measurement distributions

In [ ]:
plot_histogram(
    [row['counts'] for row in ideal_rows],
    legend=[row['circuit'] for row in ideal_rows],
    title='Measurement distributions for gate perturbations',
)
plt.show()


## 5. Probability comparison

In [ ]:
plt.figure(figsize=(11, 5))
plt.bar(ideal_df['circuit'], ideal_df['p1'])
plt.title('Probability of measuring |1>')
plt.xlabel('Circuit')
plt.ylabel('P(1)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 6. Sensitivity analysis

Two related metrics are reported:

- **Probability sensitivity:** absolute change in P(1) relative to the baseline.
- **Distribution distance:** total variation distance between the full baseline and modified output distributions.

For a one-qubit binary distribution these values are closely related, but the distribution-level metric generalizes to larger output spaces.

In [ ]:
plt.figure(figsize=(11, 5))
plt.bar(ideal_df['circuit'], ideal_df['probability_sensitivity'])
plt.title('Circuit sensitivity based on change in P(1)')
plt.xlabel('Circuit')
plt.ylabel('Absolute probability change')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
plt.bar(ideal_df['circuit'], ideal_df['distribution_distance'])
plt.title('Distribution distance from baseline')
plt.xlabel('Circuit')
plt.ylabel('Total variation distance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 7. Noise experiment

A depolarizing noise model is applied after supported quantum gates. This section studies how the measured outputs change as the noise probability increases.

In [ ]:
noise_levels = [0.00, 0.01, 0.03, 0.05, 0.10]
noise_rows = create_noise_levels(
    experiments, noise_levels, shots=SHOTS, seed=SEED
)
noise_df = pd.DataFrame([
    {k: v for k, v in row.items() if k not in ['counts', 'circuit_object']}
    for row in noise_rows
])
noise_df.head()


In [ ]:
plt.figure(figsize=(10, 6))
for circuit_name in noise_df['circuit'].unique():
    subset = noise_df[noise_df['circuit'] == circuit_name]
    plt.plot(subset['noise_probability'], subset['probability_sensitivity'], marker='o', label=circuit_name)
plt.title('Sensitivity across depolarizing noise levels')
plt.xlabel('Noise probability')
plt.ylabel('Sensitivity')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 8. Experiment summary

Use the tables and plots above to discuss which perturbations produce larger or smaller changes under the tested conditions. Avoid treating one run as a universal statement about quantum hardware.